In [ ]:
import jupyter_black
import sys

jupyter_black.load()
sys.path.insert(0, "..")


from multiprocessing import Pool

import pandas as pd
import requests
from tqdm.notebook import tqdm

from pyscripts.cache_prompting import ApiConn, add_be_urls, resp_pipe

In [ ]:
conn = ApiConn()

In [ ]:
resdf = conn.get_resdf()

In [ ]:
tid_df = conn.get_tid_df()

In [ ]:
def get_test_resps(n_procs=10, n_urls=1000, url_prefix="https://alpha.rankless.org"):
    test_urls = (
        resdf.sample(n_urls)[["rt", "semanticId"]]
        .apply(lambda r: "/".join([url_prefix, *r]), axis=1)
        .tolist()
    )
    return get_resp_df(test_urls, n_procs)


def get_resp_df(test_urls, n_procs):
    test_pool = Pool(n_procs)
    test_resps = test_pool.map(resp_pipe, test_urls)
    test_pool.terminate()
    test_pool.join()
    return pd.DataFrame(test_resps, columns=["r", "t", "s", "url"]).assign(
        nprocs=n_procs, nurls=len(test_urls)
    )

In [ ]:
def eval_tdf(df):
    return df.groupby(pd.cut(df["s"] / 1e6, [0, 0.4, 1.5, 10]), observed=True).apply(
        lambda _df: pd.Series(
            {
                **_df["t"].quantile([0.5, 0.9, 0.99, 0.999]),
                "200rate": (_df["r"] == 200).mean(),
                "count": _df.shape[0],
            }
        )
    )

In [ ]:
be_trdf = get_resp_df(
    resdf.merge(tid_df).sample(5_000).pipe(add_be_urls)["url"].tolist(), 10
)

In [ ]:
eval_tdf(be_trdf)

In [ ]:
trdf

In [ ]:
trdf = get_test_resps(10, 5_00, url_prefix="https://www.rankless.org")

In [ ]:
resdf.assign(url=lambda df: "/" + df["rt"] + "/" + df["semanticId"])[
    ["url", "citations", "papers", "rt"]
].merge(trdf.assign(url=lambda df: df["url"].str.split(".org").str[-1])).drop(
    ["url", "rt"], axis=1
).corr()